# Lab 2, Training

In [1]:
from typing import Dict, Tuple
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import models, transforms
from torchvision.utils import save_image, make_grid
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import numpy as np
from IPython.display import HTML

from dataset import CelebAHQDataset, transform
from model import DDPMUnet
from utils import *

In [2]:
device = torch.device("cuda:0" if torch.cuda.is_available() else torch.device('cpu'))
config = load_config('config.yaml')

# Setting Things Up

Section 4: Experiments of the paper, the authors state
"We set the forward process variances to constants increasing linearly from $\beta_1 = 10^{-4}$ to $\beta_T = 0.02$."

In Section 2 (Background), slightly below equation (2), the authors define a new variable $\alpha_t$ to simplify the math: $$ \alpha_t := 1 - \beta_t $$

This calculates $\bar{\alpha}_t$ (alpha-bar), which is the cumulative product of all $\alpha$ values up to timestep $t$. In the paper, it is defined as: $$ \bar{\alpha}t := \prod{s=1}^t \alpha_s $$

Why use cumsum and exp instead of cumprod? If you multiply many numbers slightly less than 1 together (like $\alpha$), the value quickly approaches 0, leading to numerical underflow. By utilizing the logarithmic property $\log(\prod x) = \sum \log(x)$, the code calculates the sum of the logs (cumsum(a_t.log())) and then exponentiates it (.exp()) to get back the product. This makes the calculation numerically stable.

These variables ($\alpha_t$ and $\bar{\alpha}_t$) are the core of Equation (4) from the paper: $$ q(x_t | x_0) = \mathcal{N}(x_t; \sqrt{\bar{\alpha}_t}x_0, (1 - \bar{\alpha}_t)\mathbf{I}) $$

Using the ab_t you just calculated, this equation gives you a magical shortcut: instead of taking thousands of small steps to add noise to an image step-by-step, you can jump directly from a clean image ($x_0$) to the noisy image at any arbitrary timestep $t$ ($x_t$) in a single step using the formula: $$ x_t = \sqrt{\bar{\alpha}_t}x_0 + \sqrt{1 - \bar{\alpha}_t}\epsilon $$

(Where $\epsilon$ is random normal noise $\mathcal{N}(0, I)$)

In [8]:
# construct DDPM noise schedule
b_t = (config['beta2'] - config['beta1']) * torch.linspace(0, 1, config['timesteps'] + 1, device=device) + config['beta1']
a_t = 1 - b_t
ab_t = torch.cumsum(a_t.log(), dim=0).exp()    
ab_t[0] = 1

# construct model
nn_model = DDPMUnet(in_channels=3, n_feat=config['n_feat'], n_cfeat=config['n_cfeat'], height=config['height']).to(device)

# perturbs an image to a specified noise level
def perturb_input(x, t, noise):
    return ab_t.sqrt()[t, None, None, None] * x + (1 - ab_t[t, None, None, None]) * noise

# Training

In [9]:
# training without context code

# load dataset and construct optimizer
# Using CelebA-HQ dataset with male/female context
dataset = CelebAHQDataset(config['data_path'], mode='train', transform=transform, null_context=True)
dataloader = DataLoader(dataset, batch_size=config['batch_size'], shuffle=True, num_workers=1)
optim = torch.optim.Adam(nn_model.parameters(), lr=config['lrate'])
nn_model.train()

Found 28000 images in /mnt/f/Datasets/CelebAHQ/train
Classes: ['female', 'male']
Class to index mapping: {'female': 0, 'male': 1}
  - female: 17943
  - male: 10057


DDPMUnet(
  (init_conv): ResidualConvBlock(
    (conv1): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): GELU(approximate='none')
    )
    (conv2): Sequential(
      (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): GELU(approximate='none')
    )
  )
  (down1): UnetDown(
    (model): Sequential(
      (0): ResidualConvBlock(
        (conv1): Sequential(
          (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): GELU(approximate='none')
        )
        (conv2): Sequential(
          (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (1): BatchNorm2d(64, eps=1e-05, 

In [ ]:
for ep in range(config['n_epoch']):
    print(f'epoch {ep}')
    
    # linearly decay learning rate
    optim.param_groups[0]['lr'] = config['lrate']*(1-ep/config['n_epoch'])
    
    pbar = tqdm(dataloader, mininterval=2 )
    for x, c in pbar:
        optim.zero_grad()
        x = x.to(device)
        c = c.to(device)

        # randomly mask out c
        context_mask = torch.bernoulli(torch.zeros(c.shape[0]) + 0.9).to(device)
        c = c * context_mask.unsqueeze(-1)
        
        # perturb data
        noise = torch.randn_like(x)
        t = torch.randint(1, config['timesteps'] + 1, (x.shape[0],)).to(device) 
        x_pert = perturb_input(x, t, noise)
        
        # use network to recover noise
        pred_noise = nn_model(x_pert, t / config['timesteps'], c)
        
        # loss is mean squared error between the predicted and true noise
        loss = F.mse_loss(pred_noise, noise)
        loss.backward()
        
        optim.step()

    # save model periodically
    if ep%4==0 or ep == int(config['n_epoch']-1):
        if not os.path.exists(config['save_dir']):
            os.mkdir(config['save_dir'])
        torch.save(nn_model.state_dict(), config['save_dir'] + f"model_{ep}.pth")
        print('saved model at ' + config['save_dir'] + f"model_{ep}.pth")

# Sampling

In [ ]:
# helper function; removes the predicted noise (but adds some noise back in to avoid collapse)
def denoise_add_noise(x, t, pred_noise, z=None):
    if z is None:
        z = torch.randn_like(x)
    noise = b_t.sqrt()[t] * z
    mean = (x - pred_noise * ((1 - a_t[t]) / (1 - ab_t[t]).sqrt())) / a_t[t].sqrt()
    return mean + noise

In [ ]:
# sample using standard algorithm with optional context
@torch.no_grad()
def sample_ddpm(n_sample, context=None, save_rate=20):
    """
    Sample from the diffusion model
    Args:
        n_sample: number of samples to generate
        context: optional context vector (e.g., [1,0] for female, [0,1] for male)
                 if None, generates unconditionally
        save_rate: how often to save intermediate samples
    """
    # x_T ~ N(0, 1), sample initial noise
    samples = torch.randn(n_sample, 3, config['height'], config['height']).to(device)  

    # prepare context
    if context is not None:
        # if single context is provided, repeat it for all samples
        if len(context.shape) == 1:
            context = context.unsqueeze(0).repeat(n_sample, 1)
        c = context.to(device)
    else:
        c = None

    # array to keep track of generated steps for plotting
    intermediate = [] 
    for i in range(config['timesteps'], 0, -1):
        print(f'sampling timestep {i:3d}', end='\r')

        # reshape time tensor
        t = torch.tensor([i / config['timesteps']])[:, None, None, None].to(device)

        # sample some random noise to inject back in. For i = 1, don't add back in noise
        z = torch.randn_like(samples) if i > 1 else 0

        eps = nn_model(samples, t, c)    # predict noise e_(x_t,t)
        samples = denoise_add_noise(samples, i, eps, z)
        if i % save_rate ==0 or i==config['timesteps'] or i<8:
            intermediate.append(samples.detach().cpu().numpy())

    intermediate = np.stack(intermediate)
    return samples, intermediate

In [ ]:
# Context vectors for conditional generation
# Female: [1, 0], Male: [0, 1]
ctx_female = torch.tensor([1, 0], dtype=torch.float32)
ctx_male = torch.tensor([0, 1], dtype=torch.float32)

# Examples:
# sample_ddpm(32, context=ctx_female)  # generates 32 female faces
# sample_ddpm(32, context=ctx_male)    # generates 32 male faces
# sample_ddpm(32, context=None)        # generates unconditionally

# samples, intermediate = sample_ddpm_context(32, ctx)
# animation_ddpm_context = plot_sample(intermediate,32,4,save_dir, "ani_run", None, save=False)
# HTML(animation_ddpm_context.to_jshtml())

#### View Epoch 0 

In [ ]:
# load in model weights and set to eval mode
nn_model.load_state_dict(torch.load(f"{config['save_dir']}/model_0.pth", map_location=device))
nn_model.eval()
print("Loaded in Model")

In [ ]:
# visualize samples
plt.clf()
samples, intermediate_ddpm = sample_ddpm(32)
animation_ddpm = plot_sample(intermediate_ddpm,32,4,config['save_dir'], "ani_run", None, save=False)
HTML(animation_ddpm.to_jshtml())

#### View Epoch 4 

In [ ]:
# load in model weights and set to eval mode
nn_model.load_state_dict(torch.load(f"{config['save_dir']}/model_4.pth", map_location=device))
nn_model.eval()
print("Loaded in Model")

In [ ]:
# visualize samples
plt.clf()
samples, intermediate_ddpm = sample_ddpm(32)
animation_ddpm = plot_sample(intermediate_ddpm,32,4,config['save_dir'], "ani_run", None, save=False)
HTML(animation_ddpm.to_jshtml())

#### View Epoch 8

In [ ]:
# load in model weights and set to eval mode
nn_model.load_state_dict(torch.load(f"{config['save_dir']}/model_8.pth", map_location=device))
nn_model.eval()
print("Loaded in Model")

In [ ]:
# visualize samples
plt.clf()
samples, intermediate_ddpm = sample_ddpm(32)
animation_ddpm = plot_sample(intermediate_ddpm,32,4,config['save_dir'], "ani_run", None, save=False)
HTML(animation_ddpm.to_jshtml())

#### View Epoch 31 

In [ ]:
# load in model weights and set to eval mode
nn_model.load_state_dict(torch.load(f"{config['save_dir']}/model_31.pth", map_location=device))
nn_model.eval()
print("Loaded in Model")

In [ ]:
# visualize samples
plt.clf()
samples, intermediate_ddpm = sample_ddpm(32)
animation_ddpm = plot_sample(intermediate_ddpm,32,4,config['save_dir'], "ani_run", None, save=False)
HTML(animation_ddpm.to_jshtml())

In [ ]:
def show_images(imgs, nrow=2):
    _, axs = plt.subplots(nrow, imgs.shape[0] // nrow, figsize=(4,2 ))
    axs = axs.flatten()
    for img, ax in zip(imgs, axs):
        img = (img.permute(1, 2, 0).clip(-1, 1).detach().cpu().numpy() + 1) / 2
        ax.set_xticks([])
        ax.set_yticks([])
        ax.imshow(img)
    plt.show()

# Acknowledgments
Sprites by ElvGames, [FrootsnVeggies](https://zrghr.itch.io/froots-and-veggies-culinary-pixels) and  [kyrise](https://kyrise.itch.io/)   
This code is modified from, https://github.com/cloneofsimo/minDiffusion   
Diffusion model is based on [Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2006.11239) and [Denoising Diffusion Implicit Models](https://arxiv.org/abs/2010.02502)
